In [ ]:
import sys
sys.path.append("/mnt/projects/sne/Gohar/12.PROJECT_Surrogate_Dynamics/surrogate-dynamics/")

from tqdm import tqdm
import numpy as np

In [ ]:
%load_ext autoreload
%autoreload 2
from dataloaders.sequence import SequenceDataModule

dm_config = {
    "input_chunk_length": 50000,
    "output_chunk_length": 0,
    "datareader_type": "XTC_CHUNKS_CG",
    "datareader_args": {
        "tprfile": "./ala2/solvated/300K/initialization/tpr_initial.tpr",
        "xtcfile": "./ala2/solvated/300K/ala2_100ns/md.xtc",
        "selection": "(resname ALA or resname ACE or resname NME) and not element H",
        "cg_window": 1,
    },
    # "dataset_type": "GRAPH",
    "dataset_type": "GRAPH_LATENT",
    "dataset_args": {
        "encoder_name": "BGE",
        "encoder_ckpt": "../train_runs/encoder/bge/run_1/checkpoints/best.ckpt",
        "precompute_graphs": False,
    },
    "train_size": 1,
    "batch_size": 1,
    "validation_size": 0,
    "val_batch_size": 0,
    "num_workers": 1,
    "n_seq_per_sample": 1,
    "norm_type": "minmax",
    "verbose": False,
}



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
dm = SequenceDataModule(**dm_config)
ds = dm.get_train_dataset()
data_points = np.array([ds[i][0].cpu().numpy() for i in tqdm(range(len(ds)))])
    

Atom index mapping (original -> selected):
  Original index: 1 -> Selected index: 0 (Element: C, Name: CH3, Type: CT, Residue: ACE1)
  Original index: 5 -> Selected index: 1 (Element: C, Name: C, Type: C, Residue: ACE1)
  Original index: 6 -> Selected index: 2 (Element: O, Name: O, Type: O, Residue: ACE1)
  Original index: 7 -> Selected index: 3 (Element: N, Name: N, Type: N, Residue: ALA2)
  Original index: 9 -> Selected index: 4 (Element: C, Name: CA, Type: CT, Residue: ALA2)
  Original index: 11 -> Selected index: 5 (Element: C, Name: CB, Type: CT, Residue: ALA2)
  Original index: 15 -> Selected index: 6 (Element: C, Name: C, Type: C, Residue: ALA2)
  Original index: 16 -> Selected index: 7 (Element: O, Name: O, Type: O, Residue: ALA2)
  Original index: 17 -> Selected index: 8 (Element: N, Name: N, Type: N, Residue: NME3)
  Original index: 19 -> Selected index: 9 (Element: C, Name: CH3, Type: CT, Residue: NME3)


Processing sequences:   0%|          | 0/3 [00:00<?, ?it/s]














































































































































































































































































































































































































































































































































































































































































































































































































































































































































































In [24]:
data_points.shape

(50000, 128)

In [ ]:
lag = 1

# Paired snapshots at lag time
X_t   = data_points[:-lag]   # shape (T-lag, d)
X_t1  = data_points[lag:]    # shape (T-lag, d)

# Instantaneous (C0) and time-lagged (Ctau) correlation matrices
C0   = X_t.T @ X_t    # (d, d)
Ctau = X_t.T @ X_t1   # (d, d)

# Transfer matrix K = C0^{-1} Ctau via least-squares (handles underdetermined case)
K, _, _, _ = np.linalg.lstsq(C0, Ctau, rcond=None)  # (d, d)

# Eigendecomposition — eigenvalues may be complex; take real parts for sorting
eigenvalues, eigenvectors = np.linalg.eig(K)

# Sort descending by magnitude (slowest modes first)
order = np.argsort(np.abs(eigenvalues))[::-1]
eigenvalues  = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

# Implied timescales in units of trajectory frames (skip stationary mode at index 0)
its = -lag / np.log(np.abs(eigenvalues[1:]))


print(f'Transfer matrix K shape: {K.shape}')
print(f'Top 10 eigenvalues (|λ|): {np.abs(eigenvalues[:10]).round(4)}')
print(f'Top 9 implied timescales (frames): {its[:9].real.round(2)}')
print(f'Eigenvectors shape: {eigenvectors.shape}')


Transfer matrix K shape: (128, 128)
Top 10 eigenvalues (|λ|): [1.     0.998  0.9618 0.924  0.8742 0.8469 0.809  0.7556 0.6985 0.6779]
Top 9 implied timescales (frames): [493.94  25.67  12.64   7.44   6.02   4.72   3.57   2.79   2.57]
Eigenfunctions shape: (49999, 128)
Eigenvectors shape: (128, 128)
